In [10]:
%pip install --upgrade --quiet ipywidgets langchain-community langchain langchain-openai faiss-cpu beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


### Define variables

In [41]:
import os

LLM_MODEL = 'gpt-4o'
TEMPERATURE = 0
K = 5
EMBEDDINGS_MODEL = 'text-embedding-ada-002'
TEXT_FIELD = 'text'
VECTOR_STORE_INDEX = 'schh'

OPENAI_API_KEY = os.getenv('OPENAPI_API_KEY')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')

### Define embeddings

In [42]:
from langchain_openai import OpenAIEmbeddings
EMBEDDINGS = OpenAIEmbeddings( model=EMBEDDINGS_MODEL, openai_api_key=OPENAI_API_KEY )

### Define LLM

In [43]:
from langchain_openai import ChatOpenAI
LLM = ChatOpenAI(model=LLM_MODEL, openai_api_key=OPENAI_API_KEY, temperature=TEMPERATURE)

### Document loaders

In [50]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

vector_store = InMemoryVectorStore(EMBEDDINGS)

def load_pdf(path, vector_store):
    loader = PyPDFLoader(path)
    docs = []
    for page in loader.load():
        docs.append(page)
    vector_store.add_documents(documents=docs)
    return docs

def load_markdown(path, vector_store):
    markdown = open(path, 'r').read()
    docs = MarkdownHeaderTextSplitter(
        headers_to_split_on = [ ('#', 'Header 1'), ('##', 'Header 2'), ('###', 'Header 3') ], 
        strip_headers=False
    ).split_text(markdown)
    vector_store.add_documents(documents=docs)
    return docs

### Use Pinecone vector store

In [4]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(VECTOR_STORE_INDEX)
vector_store = PineconeVectorStore( index, EMBEDDINGS, TEXT_FIELD )  

### Create in-memory vector store

In [59]:
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(EMBEDDINGS)

def print_docs(docs):
    for doc in docs:
        print(doc.page_content)
        print ('\n' + '-'*80 + '\n')
        
# Load documents
print_docs(load_markdown('general/2025ScheduleOfFeesAndCommunityRules.md', vector_store))
print_docs(load_markdown('groups/RegisteredCommunityGroups.md', vector_store))
    
# load_markdown('general/2025HoaFees.md', vector_store)
# load_pdf('design-guidelines/2025DesignGuidelinesSCHH.pdf', vector_store)



# 2025 Finances  
## Budget Summary 2025  
**Revenue**  
- Assessment Revenue: $25,594,464
- Reserve Contribution: $(4,803,792)
- Other Revenue: $8,141,227
- Golf Revenue: $6,081,326
- **Total Revenue**: $35,013,225
- Cost of Sales: $2,259,458
- **Total Net Revenue**: $32,753,767  
**Expenses**  
- Payroll: $6,838,280
- Utilities: $1,474,100
- Repair and Maintenance: $2,409,518
- Operating Expenses: $15,676,670
- Taxes, Licenses and Fees: $158,970
- Golf Expenses: $6,061,809
- **Total Expenses**: $32,619,347  
**Income from Operations**: $134,420  
Does not include non-cash expenses, such as depreciation

--------------------------------------------------------------------------------

## Assessment Allocation 2025  
- Operations: $2,184
- Reserve: $504
- **Total Assessment Budgeted**: $2,688

--------------------------------------------------------------------------------

## Additional Income and Expenditures  
Community Enhancement Fee
Income (*contributed directly to reserve*): $1,

### Create retriever

In [54]:
retriever = vector_store.as_retriever(search_kwargs={'k': K})

### Query and print retrieved docs

In [52]:
question = 'What are registered community groups?'

In [ ]:
import json


for doc in retriever.invoke(question):
    print (doc.page_content.replace('\n', ' ') + '\n')
    print (json.dumps(doc.metadata))
    print ('\n' + '-'*80 + '\n')

In [60]:
for doc in vector_store.similarity_search(query=question, k=K):
    print (doc.page_content.replace('\n', ' ') + '\n')
    print (json.dumps(doc.metadata))
    print ('\n' + '-'*80 + '\n')

## 26. REGISTERED COMMUNITY GROUPS   Registered Community Groups are groups that provide additional opportunities for residents to come together with mutual interests that further enhance their lives and the lifestyle of the community. These groups may form as they cannot retain charter status or elect to remain unchartered. Registered Community Groups may be formed around social, service, political, geographical, vocational, or educational interests. See a list of Registered Community Groups in each issue of SunSations.

{"Header 1": "COMMUNITY RULES 2025", "Header 2": "26. REGISTERED COMMUNITY GROUPS"}

--------------------------------------------------------------------------------

# Registered Community Groups   [100+ SC Women Who Care](https://suncityhiltonhead.org/GroupPage/46134~2515)   [AARP Tax Aide Program](https://suncityhiltonhead.org/GroupPage/46134~2592)   [Alzheimers/Dementia Support Group](https://suncityhiltonhead.org/GroupPage/46134~4405)   [American Revolution Round